In [ ]:
!pip install -q -U transformers datasets bitsandbytes  trl peft  huggingface_hub

In [1]:
from huggingface_hub import login,whoami
login()
print(whoami())

{'type': 'user', 'id': '69d4675c26493c5725f3e1ea', 'name': 'Chufeng-Jiang', 'fullname': 'Chufeng Jiang', 'email': 'jiangchufengjcf@gmail.com', 'emailVerified': True, 'canPay': False, 'billingMode': 'prepaid', 'periodEnd': 1777593600, 'isPro': False, 'avatarUrl': 'https://cdn-avatars.huggingface.co/v1/production/uploads/69d4675c26493c5725f3e1ea/pY8UdnpJSc0loAQ9Ac_qi.jpeg', 'orgs': [], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'test', 'role': 'write', 'createdAt': '2026-04-07T02:12:24.103Z'}}}


In [4]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModelForCausalLM
import torch

model_name = "mistralai/Mistral-7B-v0.3"

config_4bit = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

model_4bit = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=config_4bit,
    device_map="auto",
    low_cpu_mem_usage=True,
    max_memory={0: "12GiB", "cpu": "48GiB"},
    offload_folder="offload",
    trust_remote_code=True
)

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    padding_side="left",
    trust_remote_code=True
)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

/home/chufeng/SoftwareInstalled/anaconda3/envs/LLM/lib/python3.10/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

In [5]:
print("Tokenizer loaded:", tokenizer is not None)
print("Vocab size:", len(tokenizer))

Tokenizer loaded: True
Vocab size: 32768


In [6]:
# from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModelForCausalLM
# import torch

# model_name = "mistralai/Mistral-7B-v0.3"
# config_4bit = BitsAndBytesConfig(load_in_4bit=True)

# model_4bit = AutoModelForCausalLM.from_pretrained(
#                                                   model_name,
#                                                   quantization_config=config_4bit,
#                                                   device_map="auto",
#                                                   trust_remote_code=True
# )

# tokenizer =AutoTokenizer.from_pretrained(model_name,padding_side="left",trust_remote_code=True)

In [7]:
from datasets import load_dataset

dataset = load_dataset("MattCoddity/dockerNLcommands")
dataset

DatasetDict({
    train: Dataset({
        features: ['input', 'output', 'instruction'],
        num_rows: 2415
    })
})

In [8]:
tokenizer.all_special_tokens

['<s>', '</s>', '<unk>']

In [9]:
tokens = ['<system>', '<user>', '<assistant>']

for token in tokens:
    token_id = tokenizer.encode(token,add_special_tokens=False)
    print(token,token_id)

<system> [1291, 7342, 29535]
<user> [1291, 2606, 29535]
<assistant> [1291, 1257, 11911, 29535]


In [10]:
tokenizer.pad_token

In [11]:
special_token = {
                 'pad_token': '<PAD>',
                 'additional_special_tokens': ['<system>', '<user>', '<assistant>']
}

tokenizer.add_special_tokens(special_token)

4

In [12]:
tokens = ['<system>', '<user>', '<assistant>','<PAD>']

for token in tokens:
    token_id = tokenizer.encode(token,add_special_tokens=False)
    print(token,token_id)

<system> [32769]
<user> [32770]
<assistant> [32771]
<PAD> [32768]


In [13]:
len(tokenizer)

32772

In [14]:
model_4bit.config.vocab_size

32768

In [15]:
model_4bit.resize_token_embeddings(len(tokenizer))

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
The new lm_head weights will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Embedding(32772, 4096)

In [16]:
tokenizer.all_special_tokens

['<s>', '</s>', '<unk>', '<PAD>', '<system>', '<user>', '<assistant>']

In [17]:
tokenizer.bos_token

'<s>'

In [18]:
tokenizer.eos_token

'</s>'

In [19]:
dataset['train'][0]

{'input': 'Give me a list of containers that have the Ubuntu image as their ancestor.',
 'output': "docker ps --filter 'ancestor=ubuntu'",
 'instruction': 'translate this sentence in docker command'}

In [20]:
new_template = """<s><system>{system_prompt}</s><user>{user_prompt}</s><assistant>{model_answer}</s>"""

def format_dataset(example):

    system_prompt = example["instruction"]
    user_prompt = example["input"]
    model_answer = example["output"]

    formatted_text = new_template.format(

                                        system_prompt= system_prompt,
                                        user_prompt = user_prompt,
                                        model_answer =model_answer
    )

    return {"text": formatted_text}

format_dataset(dataset['train'][0])

{'text': "<s><system>translate this sentence in docker command</s><user>Give me a list of containers that have the Ubuntu image as their ancestor.</s><assistant>docker ps --filter 'ancestor=ubuntu'</s>"}

In [21]:
dataset = dataset.map(format_dataset)
dataset['train']['text'][0]

Map:   0%|          | 0/2415 [00:00<?, ? examples/s]

"<s><system>translate this sentence in docker command</s><user>Give me a list of containers that have the Ubuntu image as their ancestor.</s><assistant>docker ps --filter 'ancestor=ubuntu'</s>"